
<div class='alert alert-info' style='text-align:justify'>

This notebook is used to explore Gaussian Mixture Models monitors (GMM Monitors) and see how the different parameters influence the performance of the monitoring task. A few metrics are computed to briefly compare the different versions of monitors.
</div>

## Getting hands on

In [1]:
from dataset import Dataset
from feature_extractor import FeatureExtractor
from evaluator import Evaluator
from evaluator import compute_aupr, compute_auroc, compute_tnr_frac_tpr

from Monitors import GMMMonitor
from Monitors import MahalanobisMonitor

import csv
import os
import pandas as pd
import torch

In [2]:
model = "densenet"
layer = 98

dataset_ID  = "cifar10"
dataset_OOD = "cifar100"

perturbation = None
adver_attack = None

In [3]:
batch_size = 100
TORCH_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [ ]:
dataset_train = Dataset(dataset_ID, "train", model, batch_size=batch_size)
dataset_test = Dataset(dataset_ID, "test", model, batch_size=batch_size)
dataset_ood = Dataset(dataset_OOD, "test", model, perturbation, adver_attack, batch_size=batch_size)

In [ ]:
feature_extractor = FeatureExtractor(model, dataset_ID, [layer], TORCH_DEVICE)

In [6]:
# deep_features_train = feature_extractor.get_features(dataset_train)
# deep_features_test = feature_extractor.get_features(dataset_test)
# deep_features_ood = feature_extractor.get_features(dataset_ood)

features_train, logits_train, softmax_train, \
    preds_train, labels_train = feature_extractor.get_features(dataset_train)
features_test, logits_test, softmax_test, \
    preds_test, labels_test = feature_extractor.get_features(dataset_test)
features_ood, logits_ood, softmax_ood, \
    preds_ood, labels_ood = feature_extractor.get_features(dataset_ood)

In [7]:
eval_oms = Evaluator("oms", is_novelty=(dataset_ID != dataset_OOD))
eval_oms.fit_ground_truth(labels_test, labels_ood, preds_test, preds_ood)

eval_ood = Evaluator("ood", is_novelty=(dataset_ID != dataset_OOD))
eval_ood.fit_ground_truth(labels_test, labels_ood, preds_test, preds_ood)

In [8]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
hp_n_components = ['auto_aic', 'auto_bic', 'auto_knee']
hp_t_covariance = ['full', 'diag', 'tied', 'spherical']

result = {}

for n in hp_n_components:
    for c in hp_t_covariance:
        print(f"... evaluating parameters (n={n}, c={c})")
        
        gmm_monitor = GMMMonitor(dataset_ID, model, layer, n_components=n, t_covariance=c)
        gmm_monitor.fit(features_train[0], preds_train, labels_train, save=True)
        gmm_name = "GMM_%s_%s" % (n, c)

        scores_test = gmm_monitor.predict(features_test[0], preds_test)
        scores_ood  = gmm_monitor.predict(features_ood[0], preds_ood)
        
        # Compute global metrics under OOD evaluation paradigm
        aupr_OOD  = eval_ood.get_aupr_score(scores_test, scores_ood)
        auroc_OOD = eval_ood.get_auroc_score(scores_test, scores_ood)
        tnr95_OOD = eval_ood.get_tnr_frac_tpr(scores_test, scores_ood, 0.95)

        # Compute global metrics under OMS evaluation paradigm
        aupr_OMS = eval_oms.get_aupr_score(scores_test, scores_ood)
        auroc_OMS = eval_oms.get_auroc_score(scores_test, scores_ood)
        tnr95_OMS = eval_oms.get_tnr_frac_tpr(scores_test, scores_ood, 0.95)

        data = [
            aupr_OOD, auroc_OOD, tnr95_OOD,
            aupr_OMS, auroc_OMS, tnr95_OMS,
        ]
        result[gmm_name] = data

In [ ]:
df = pd.DataFrame.from_dict(result, orient='index')
df

## Experiment

Run the `Scripts/explo_gmm_script.py` script to get performance results of the GMM monitor with diverse hyper-parameters.

In [ ]:
%run Scripts/explo_gmm_script.py